In [ ]:
# wymagane importy

from bokeh.io import output_notebook
from omni_drive import (
    Angle,
    Clock,
    Distance,
    DriveVector,
    OmniDrive,
    PoseRecorder,
    SensorRecorder,
    Time,
)
from omni_drive.ui import LivePlot

output_notebook(hide_banner=True)


# OmniDrive API

<small>

Poniższy notebook przedstawia interfejs API do sterowania robotem OmniDrive oraz prezentuje jego możliwości.

</small>

### Konfiguracja połączenia z robotem:

In [2]:
robot = OmniDrive()

### Sposoby sterowania robotem OmniDrive

<small>

Biblioteka `omni_drive` udostępnia dwa główne paradygmaty sterowania platformą:

1. **Tryb reaktywny (mikro-kroki)** – pętla czasu rzeczywistego ze sprzężeniem zwrotnym. Pozwala na żywo reagować na odczyty czujników, płynnie regulować prędkość i realizować profile ruchu.
2. **Tryb deklaratywny (makro-kroki)** – sekwencje gotowych manewrów w trybie blokującym. Wykonujesz z góry zaplanowane ruchy, czekając na ukończenie każdego z nich przed przejściem dalej.

</small>

#### 1. Tryb reaktywny

<small>

Pętla taktowana przez zegar narzuca stały takt czasowy, samodzielnie dba o uśpienia.

</small>

In [3]:
async for t in Clock():
    # Zadajemy wektor prędkości (jazda do przodu z prędkością 0.2 m/s):
    robot.set_vector(DriveVector(velocity=0.2, heading=Angle.deg(0)))

    # Warunek czasowy – zatrzymanie po 2 sekundach:
    if t > 2.0:
        robot.stop()
        break


**Odczyt czujników:**

<small>

W pętli odpytujemy bezpośrednio właściwości robota i reagujemy na bieżąco.

</small>

In [ ]:
try:
    async for _ in Clock():
        # Odczytujemy stan czujnika bezpośrednio z robota:
        dist_front = robot.us_2.distance

        # Zatrzymanie gdy przeszkoda < 20 cm:
        if dist_front < Distance.cm(20):
            print("Wykryto przeszkodę - zatrzymanie!")
            break

        robot.set_vector(DriveVector(velocity=0.2, heading=Angle.deg(0)))
finally:
    robot.stop()


**Współdzielenie zegara:**

<small>

Wspólna instancja `Clock` dyktuje tempo próbkowania dla innych komponentów (np. `SensorRecorder`) i synchronizuje rejestrację danych z pętlą robota.

</small>


In [ ]:
clock = Clock()

# Definiujemy źródła danych powiązane z zegarem i uruchamiamy wykres na żywo:
rec = SensorRecorder(
    clock,
    front=lambda: robot.us_2.distance.meters,
    left=lambda: robot.us_1.distance.meters,
)
live = LivePlot(rec).show()

# Pętla sterowania korzysta z tego samego zegara:
try:
    async for t in clock:
        robot.set_vector(DriveVector(velocity=0.2, heading=Angle.deg(0)))

        if t > 3.0:
            break
finally:
    robot.stop()

live.stop()


#### 2. Tryb deklaratywny

<small>

Tryb deklaratywny jest zwięzły i świetnie sprawdza się przy z góry zaplanowanych manewrach. Ten tryb jest blokujący – program czeka na zakończenie każdego zadanego ruchu przed przejściem do kolejnej instrukcji:

</small>

In [ ]:
# 1. Jazda do przodu przez 2 sekundy:
await robot.drive_for(Time.s(2), DriveVector(velocity=0.2, heading=Angle.deg(0)))

# 2. Postój przez 1 sekundę:
await robot.pause_for(Time.s(1))

# 3. Obrót platformy w miejscu przez 2 sekundy (omega = prędkość kątowa):
await robot.drive_for(Time.s(2), DriveVector(velocity=0.0, heading=Angle(0), omega=0.5))


### Pozycja robota i zerowanie odometrii

<small>

Sterownik na bieżąco wyznacza pozycję robota na podstawie enkoderów (`robot.pose`). W każdej chwili względną pozycję robota, która jest trzymana w pamięci programu, można zresetować:

</small>


In [ ]:
print(robot.pose.x, robot.pose.y, robot.pose.yaw)  # np. 0.45m  0.12m  35.0°

robot.reset_pose()

print(robot.pose.x, robot.pose.y, robot.pose.yaw)  # 0.0m  0.0m  0.0°


### Jednostki wielkości fizycznych (Distance, Angle, Time)

<small>

Biblioteka `omni_drive` używa dedykowanych typów do reprezentowania wielkości fizycznych, zapobiegając pomyłkom jednostek:

* **Odległość (`Distance`)**: tworzenie przez `Distance.m()`, `Distance.cm()`, `Distance.mm()`. Odczyt wartości w metrach (`.meters`), centymetrach (`.as_cm`) lub milimetrach (`.as_mm`).
* **Kąt (`Angle`)**: tworzenie przez `Angle.deg()`, `Angle.rad()`. Odczyt w stopniach (`.degrees`) lub radianach (`.radians`).
* **Czas (`Time`)**: tworzenie przez `Time.s()`, `Time.ms()`. Odczyt w sekundach (`.seconds`).

</small>

In [ ]:
# Przykłady tworzenia i odczytu jednostek:
dist = Distance.cm(25)
print(f"Odległość: {dist.meters} m = {dist.as_cm} cm = {dist.as_mm} mm")

ang = Angle.deg(90)
print(f"Kąt: {ang.degrees:.1f}° = {ang.radians:.3f} rad")

t = Time.s(1.5)
print(f"Czas: {t.seconds} s")


### Czujniki odległości (IR oraz ultradźwiękowe)

<small>

Platforma jest wyposażona w dwa zestawy czujników odległości rozmieszczonych po obwodzie:

* **Czujniki ultradźwiękowe (US):**
  * `robot.us_1` – lewy (+90°)
  * `robot.us_2` – środkowy / przedni (0°)
  * `robot.us_3` – prawy (-90°)
* **Czujniki podczerwieni (IR):**
  * `robot.ir_1` – lewy (+90°)
  * `robot.ir_2` – środkowy / przedni (0°)
  * `robot.ir_3` – prawy (-90°)

Właściwość `.distance` każdego czujnika zwraca obiekt typu `Distance`.

</small>

In [ ]:
# Odczyt odległości z czujników pokładowych:
print(f"Przód:  IR = {robot.ir_2.distance.meters:.3f} m  |  US = {robot.us_2.distance.meters:.3f} m")
print(f"Lewo:   IR = {robot.ir_1.distance.meters:.3f} m  |  US = {robot.us_1.distance.meters:.3f} m")
print(f"Prawo:  IR = {robot.ir_3.distance.meters:.3f} m  |  US = {robot.us_3.distance.meters:.3f} m")


### Skaner laserowy LiDAR

<small>

Skaner laserowy LiDAR udostępnia uśrednione wiązki kierunkowe w osiach sensorów platformy:

* `robot.lidar.beam_1` – wiązka w lewo (+90°)
* `robot.lidar.beam_2` – wiązka na wprost (0°)
* `robot.lidar.beam_3` – wiązka w prawo (-90°)

Wiązki zwracają obiekt typu `Distance`.

</small>

In [ ]:
# Wiązki kierunkowe ze skanera laserowego LiDAR:
print(f"LiDAR lewy (+90°):  {robot.lidar.beam_1.meters:.3f} m")
print(f"LiDAR przód (0°):   {robot.lidar.beam_2.meters:.3f} m")
print(f"LiDAR prawy (-90°): {robot.lidar.beam_3.meters:.3f} m")


### Pozostałe funkcjonalności

<small>

Elementy telemetrii i diagnostyki, które zazwyczaj nie są wykorzystywane bezpośrednio przy podstawowym sterowaniu ruchem:

* `robot.power` – parametry zasilania platformy (`percentage`, `voltage`, `current`, `is_critical`),
* `robot.lidar` – surowy skan ze skanera laserowego; `robot.lidar.point_cloud` zwraca tablicę NumPy (`N x 2`) punktów `[x, y]` w metrach (wykorzystywaną wewnętrznie m.in. przez moduł SLAM),
* `robot.print_status()` – wypisanie pełnego podsumowania diagnostycznego stanu robota w konsoli.

</small>

In [ ]:
# Parametry zasilania:
print(f"Bateria: {robot.power.percentage:.1f}% ({robot.power.voltage:.2f} V)")
if robot.power.is_critical:
    print("Ostrzeżenie: niski poziom naładowania akumulatora!")

# Surowa chmura punktów skanera LiDAR (wykorzystywana m.in. w SLAM):
pts = robot.lidar.point_cloud
print(f"Liczba punktów w chmurze LiDAR: {len(pts)}")

# Całościowy raport stanu robota wypisywany w konsoli:
robot.print_status()


### Układ odniesienia (robot.field_centric)

<small>

Podczas zadawania wektora prędkości (`DriveVector`) kierunek ruchu (`heading`) może być interpretowany w jednym z dwóch układów odniesienia w obu trybach:

* `robot.field_centric = True` (**Field-Centric, domyślny**) – kierunek liczony jest w globalnym układzie odniesienia (układzie świata). Kąt 0° to zawsze ten sam stały kierunek w pokoju, nawet jeśli robot obraca się wokół własnej osi.
* `robot.field_centric = False` (**Robot-Centric**) – kierunek liczony jest w lokalnym układzie platformy. Kąt 0° to zawsze kierunek, w który skierowany jest przód robota (`us_2`). Gdy robot się obraca, kierunki jazdy obracają się razem z nim.

Flagę można przełączać w dowolnym momencie w trakcie pracy.

</small>

**Układ odniesienia - świat:**

In [ ]:
# Domyślnie włączony jest tryb Field-Centric – kierunek zadany w globalnym układzie świata:
print(robot.field_centric) # True
# 1. Ruch w kierunku 0°
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))
# 2. Obrót o 90° stopni
await robot.rotate_for(Angle.deg(90), speed=0.5)
# 2. Ruch dalej w kierunku 0° POMIMO ZMIANY ZWROTU PLATFORMY
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))

**Układ odniesienia - nos robota:**

In [ ]:
# Przełączenie na tryb Robot-Centric – kierunek zadany względem przodu robota:
robot.field_centric = False
# 1. Ruch w kierunku 0°
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))
# 2. Obrót o 90° stopni
await robot.rotate_for(Angle.deg(90), speed=0.5)
# 3. Ruch w kierunku 90° ZGODNIE Z TYM GDZIE ZWRÓCONY JEST TERAZ ROBOT
await robot.drive_for(Time.s(1), DriveVector(velocity=0.2, heading=Angle.deg(0)))

W zależności od tego ustawienia różnić się będzie m.in. zachowanie w ruchu złożonym. Tzn. takim, w którym robot obraca się podczas jazdy:

In [ ]:
await robot.drive_for(Time.s(1), DriveVector(velocity=0.4, heading=Angle.deg(0), omega=0.5))

![Porównanie trybów Field-Centric i Robot-Centric](assets/field_vs_robot_centric.svg)